## MCP项目实践
LangChain调用MCP是可以将MCP的工具直接转换为LangChain的工具，然后通过预定义的MCP_Client实现与外部MCP的读写操作，换而言之就是我们需要改写原先的client，将原先的Function calling调用逻辑修改为LangChain调用逻辑

### 创建 mcp server

In [ ]:
%pip install FastMCP

In [1]:
import json
import os
import httpx
import dotenv
from loguru import logger
from mcp.server.fastmcp import FastMCP

dotenv.load_dotenv()

# =========================
# 创建 MCP Server（SSE模式）
# =========================
mcp = FastMCP(
    name="WeatherServerSSE",
    host="0.0.0.0",
    port=8000
)


# =========================
# 工具：天气查询
# =========================
@mcp.tool()
def get_weather(city: str) -> str:
    """
    查询指定城市的即时天气信息（OpenWeather API）

    参数:
        city: 城市英文名，如 Beijing / Shanghai

    返回:
        JSON字符串（天气数据）
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }

    # 调用外部API
    resp = httpx.get(url, params=params, timeout=10)
    data = resp.json()

    logger.info(f"[MCP] {city} 天气查询完成")

    return json.dumps(data, ensure_ascii=False)


# =========================
# 启动 MCP SSE Server
# =========================
if __name__ == "__main__":
    logger.info("🚀 MCP Weather Server 启动中...")
    logger.info("📡 SSE地址: http://0.0.0.0:8000/sse")

    # SSE模式启动（MCP标准传输方式）
    mcp.run(transport="sse")

2026-05-26 13:52:31.795 | INFO     | __main__:<module>:57 - 🚀 MCP Weather Server 启动中...
2026-05-26 13:52:31.795 | INFO     | __main__:<module>:58 - 📡 SSE地址: http://0.0.0.0:8000/sse


RuntimeError: Already running asyncio in this thread

### 创建 mcp配置文件
创建mcp.json文件填写以下内容

In [ ]:
{
  "mcpServers": {
    "weather": {
      "url": "http://127.0.0.1:8000/sse",
      "transport": "sse"
    }
  }
}

### Langchain 客户端

In [ ]:
%pip install langchainhub

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
from loguru import logger
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)


# =========================
# 读取 MCP 配置
# =========================
def load_servers(file_path="mcp.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)["mcpServers"]


# =========================
# 主程序
# =========================
async def main():

    # 1️⃣ 加载 MCP Server配置
    servers = load_servers()

    # 2️⃣ 创建 MCP Client
    client = MultiServerMCPClient(servers)

    # 3️⃣ 获取 MCP Tools（自动发现）
    tools = await client.get_tools()
    logger.info(f"🧩 已加载工具: {[t.name for t in tools]}")

    # 4️⃣ 初始化 LLM（DeepSeek在线模型）
    llm = init_chat_model(
        "deepseek-chat",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY")  
    )

    # 5️⃣ 创建 Agent（🔥关键替换点）
    agent = create_react_agent(llm, tools)

    # 6️⃣ CLI循环
    logger.info("🤖 MCP Agent 已启动，输入 quit 退出")

    while True:
        query = input("\n你: ").strip()

        if query.lower() == "quit":
            break

        try:
            # 🚀 LangGraph调用方式
            result = await agent.ainvoke({
                "messages": [
                    ("user", query)
                ]
            })

            # 最终输出
            final_msg = result["messages"][-1].content
            print("\nAI:", final_msg)

        except Exception as e:
            logger.error(f"❌ 错误: {e}")


# =========================
# 启动入口
# =========================
if __name__ == "__main__":
    asyncio.run(main())